In [1]:
import torch


In [2]:
a = torch.arange(24)
b = a.reshape(2,3,4)
b

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

In [3]:
c = b[1:2]
c

tensor([[[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

In [4]:
print(f"{a.storage_offset()=}")
print(f"{c.storage_offset()=}")
print(f"{a.stride()=}")
print(f"{b.stride()=}")
print(f"{a.data_ptr()=}")
print(f"{b.data_ptr()=}")
print(f"{a.device=}")
print(f"{b.device=}")

a.storage_offset()=0
c.storage_offset()=12
a.stride()=(1,)
b.stride()=(12, 4, 1)
a.data_ptr()=2828638683456
b.data_ptr()=2828638683456
a.device=device(type='cpu')
b.device=device(type='cpu')


In [5]:
c = a.reshape(4,6)
c = c.t()

In [6]:
print(f"{a.data_ptr()=}")
print(f"{c.data_ptr()=}")
# print()

d = c.contiguous() # => returns copy since this data can't be arranged as continguous view
print(f"{d.data_ptr()=}")
print(d.stride())
print(c.stride())

a.data_ptr()=2828638683456
c.data_ptr()=2828638683456
d.data_ptr()=2828638683648
(4, 1)
(1, 6)


In [7]:
print(f"{c.is_contiguous()=}")
c.view(-1) # asking for a 1 d view , which not possible, but a reshape copies data and makes it possible

c.is_contiguous()=False


RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [8]:
e = c.reshape(-1)
print(e)
print(f"{e.data_ptr()=}")

tensor([ 0,  6, 12, 18,  1,  7, 13, 19,  2,  8, 14, 20,  3,  9, 15, 21,  4, 10,
        16, 22,  5, 11, 17, 23])
e.data_ptr()=2828638683840


In [9]:
x = torch.arange(4)
y = x.expand((3,4)) # broadcasting (sets stride = 0 for row)
print(y)
print(f"{x.shape=}")
print(f"{y.shape=}")
print(f"{x.stride()=}")
print(f"{y.stride()=}")

tensor([[0, 1, 2, 3],
        [0, 1, 2, 3],
        [0, 1, 2, 3]])
x.shape=torch.Size([4])
y.shape=torch.Size([3, 4])
x.stride()=(1,)
y.stride()=(0, 1)


In [10]:
x = x.to("cuda")
#by defalut tensors have requres grad false
y = torch.ones_like(x, device= "cuda", requires_grad= True, dtype= torch.float32)
z = x  * y; z.retain_grad() # by defalut intermediate tensors don't retain grad, intermediate 
                            #intermediate tensors produced by op a tensor that has req grad true will also have req grad true
l = z.sum()
l.backward()

In [11]:
try:
    l.backward()
except Exception as e:
    print(e)

Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.


In [12]:
print(f"{z.grad=}")
print(f"{y.grad=}")
print(f"{x.grad=}")


z.grad=tensor([2., 2., 2., 2.], device='cuda:0')
y.grad=tensor([0., 1., 2., 3.], device='cuda:0')
x.grad=None


In [13]:
print(f"{y.grad_fn=}")
print(f"{z.grad_fn=}")

y.grad_fn=None
z.grad_fn=<MulBackward0 object at 0x0000023A1FDCD510>


In [14]:
#tensor with  gradfn  = none is considered a leaf tensor
print(f"{z.is_leaf=}")
print(f"{y.is_leaf=}")


z.is_leaf=False
y.is_leaf=True


In [15]:
a = torch.tensor([1,2,3])
print(f"{a.device=}")
b = a.cuda()
print(f"{b.device=}")
print(b)

a.device=device(type='cpu')
b.device=device(type='cuda', index=0)
tensor([1, 2, 3], device='cuda:0')


In [16]:
t = torch.arange(12).reshape(3, 4)
t.T.view(12)                 # RuntimeError: view size is not compatible...
t.T.reshape(12)              # works — reshape() falls back to a copy when needed
t.T.contiguous().view(12)    # also works — contiguous() materializes a fresh copy first

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [3]:
import torch

In [ ]:
class Model():
    def __init__(self, nin, nout):
        self.W = torch.rand((nin, nout), requires_grad = True, device = "cuda")
        self.B = torch.rand((1, nout), requires_grad = True, device = "cuda")

    def __call__(self, data_batch, inference = False):
        if inference:
            with torch.no_grad():
                return (data_batch @ self.W) + self.B
        return (data_batch @ self.W) + self.B

    def loss(self, data_batch, y_true):
        return torch.mean(torch.square(y_true - self(data_batch))) #don't use inplace operations it will cause autograd and backprop erorrs
                                                                    #let it generate new intermediate activations
    def update(self, lr):
        with torch.no_grad():
            self.W -=  lr * self.W.grad        #in place operataion better not to record these operations (anyhow no use)
            self.B -=  lr * self.B.grad
        self.W.grad = None
        self.B.grad = None
            

In [5]:
X =  torch.rand((50, 4)) * 100
Y =  X @ (torch.tensor([2, 4, 8, 1.0]).reshape(4,1))  + 10

In [6]:
print(X[:10])
print(Y[:10])

tensor([[ 9.3582, 88.9302, 82.3615, 42.3378],
        [44.1508, 25.4646, 43.1591, 98.1039],
        [79.1976, 97.7842, 18.3210, 58.4280],
        [34.3602, 54.2397, 68.9917, 91.4677],
        [54.9012, 65.0315,  8.1321, 40.9615],
        [99.8326, 74.9367, 61.6253, 67.0083],
        [52.0001, 51.9766, 61.7287, 50.0969],
        [54.3866, 41.5242, 94.6772, 13.2351],
        [90.1427, 94.3073, 19.0575, 33.9468],
        [47.1536, 61.9828, 37.4498, 92.1716]])
tensor([[1085.6669],
        [ 643.5367],
        [ 764.5283],
        [ 939.0804],
        [ 485.9472],
        [1069.4227],
        [ 865.8330],
        [1055.5231],
        [ 753.9218],
        [ 744.0087]])


In [ ]:
# model(X, inference=True)[:10]
def update(model,steps, lr):
    for step in range(steps):
        loss = model.loss(X.cuda(), Y.cuda())
        if step % 1000 == 0:
            print(f"step_{step}:{loss=}")
        loss.backward()
        model.update(lr)
    print("W\n", model.W)
    print("B\n", model.B)
    #print("W:grad\n", model.W.grad)
    print(f"final loss :{model.loss(X.cuda(),Y.cuda())}")


In [ ]:
model_2 = Model(4, 1)
update(model_2, 100, 1e-2)

step_0:loss=tensor(499063.5938, device='cuda:0', grad_fn=<MeanBackward0>)
W
 tensor([[nan],
        [nan],
        [nan],
        [nan]], device='cuda:0', requires_grad=True)
B
 tensor([[nan]], device='cuda:0', requires_grad=True)
final loss :nan


: 

In [ ]:
model_1 = Model(4, 1)
update(model_1, 1000000, 1e-7)

step_0:loss=tensor(498870.9375, device='cuda:0', grad_fn=<MeanBackward0>)
step_100:loss=tensor(324213.1250, device='cuda:0', grad_fn=<MeanBackward0>)
step_200:loss=tensor(214042.3594, device='cuda:0', grad_fn=<MeanBackward0>)
step_300:loss=tensor(144419.6562, device='cuda:0', grad_fn=<MeanBackward0>)
step_400:loss=tensor(100297.5156, device='cuda:0', grad_fn=<MeanBackward0>)
step_500:loss=tensor(72217.3203, device='cuda:0', grad_fn=<MeanBackward0>)
step_600:loss=tensor(54232.9531, device='cuda:0', grad_fn=<MeanBackward0>)
step_700:loss=tensor(42606.8359, device='cuda:0', grad_fn=<MeanBackward0>)
step_800:loss=tensor(34989.0273, device='cuda:0', grad_fn=<MeanBackward0>)
step_900:loss=tensor(29902.0742, device='cuda:0', grad_fn=<MeanBackward0>)
step_1000:loss=tensor(26416.8652, device='cuda:0', grad_fn=<MeanBackward0>)
step_1100:loss=tensor(23949.0273, device='cuda:0', grad_fn=<MeanBackward0>)
step_1200:loss=tensor(22130.8242, device='cuda:0', grad_fn=<MeanBackward0>)
step_1300:loss=tens

In [ ]:
#since intial loss is very high the gradient will also be very high
#on the loss surface we are at a great height
#so we might overshoot if the learning rate is low
#so 1e-2 didn't work
#but 1e-7 worked
#this is why proper intialization, normalization of features, and learning rete schedules and optimizers are very important
#here the problem is with small intial predicted values due small weight intialization(which is normal)
#instead we should are normalized the features of the data to make them small and workable

In [ ]:
p = torch.randn((3, 4), requires_grad = True)
q = torch.randn((4,3))


In [ ]:
r = p @ q
s = r.sum()
print(f"{s=}")

s=tensor(1.3212, grad_fn=<SumBackward0>)


In [ ]:
s.backward()

In [ ]:
print(f"{p.grad=}")
print(f"{p.grad.shape=}")

p.grad=tensor([[-0.1031,  0.4161,  0.1211, -0.4388],
        [-0.1031,  0.4161,  0.1211, -0.4388],
        [-0.1031,  0.4161,  0.1211, -0.4388]])
p.grad.shape=torch.Size([3, 4])


In [ ]:
print(p.data.data_ptr())
print(p.data_ptr())
p is p.data # false because underlying storage is same but pyton tensor object is differt (view)


4053241497536
4053241497536


False

In [ ]:
n = torch.tensor([1, 2, 3], device= "cuda")
n.data  # n.data will be on same device as n(it's obivous right dumbass)


tensor([1, 2, 3], device='cuda:0')